# Buscar una molienda factible para magnetita

**Pregunta:** ¿Qué molienda factible selecciona la búsqueda de producción para el circuito de magnetita de autoría propia?

Nivel: 1. Requisitos previos: lecciones anteriores del tema. Lectura: 30 minutos.



## Objetivos

El circuito de magnetita fina aporta un problema unidimensional de molienda. Se registran estado y costo exacto para pesos normalizados iguales de metal y energía. Factibilidad, convergencia y acuerdo físico son resultados distintos y no se presuponen.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Importar las herramientas fijadas y resolver las entradas relativas a la carpeta del notebook.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. Las entradas son el caso `iron_magnetite_fine` del catálogo y `data/derived/contract/operating_contract.json`. No se usa un conjunto medido de planta ni modelo económico. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

La búsqueda inicial controla solo molienda de magnetita fina. El objetivo asigna 0.5 al metal respecto del nominal y 0.5 a energía específica negativa respecto de la nominal. Ley, potencia requerida y agua siguen como restricciones. La barrera progresiva multicomienzo distingue óptimo factible, extremo de menor violación y base rechazada. Se intenta auditoría separada solo para un estado definido, fuera del conteo de candidatos. La tabla relaciona molienda, factibilidad, holguras y detención real, sin asumir éxito porque exista un vector.

Partir de un circuito de magnetita con una decisión. El contrato limita molienda y el nominal entrega escalas de metal y energía. Una dimensión permite inspeccionar la búsqueda conservando restricciones reales de ley, potencia y agua. El motor calcula recuperación base, no una curva ajustada para ilustración. No hace falta barrer tratamiento para entender qué decisión controla el optimizador.

In [ ]:
# Usar el circuito de magnetita de autoría propia como búsqueda acotada de una dimensión.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.optimization import optimize, Problem
from pipeline.methods.pattern_search import pattern_search
from pipeline.engine.constants import constant

case = CASE_BY_ID["iron_magnetite_fine"]
contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
base = simulate(case.ore, case.plant, case.nominal)
print("Recuperación base (%):", round(base.metrics["recovery_pct"], 3))
print("Límites de decisión:", contract["cases"][case.id]["inputs"]["target_p80_um"])


Llamar al optimizador con pesos normalizados iguales y sin filtro. Usar los comienzos declarados y barrera progresiva para reducir violaciones. Leer estado terminal, conteo exacto y motivo de detención antes de buscar la mejor configuración. Un extremo factible no prueba óptimo global; detener por presupuesto tampoco equivale a satisfacer malla mínima.

In [ ]:
# Ejecutar la búsqueda multicomienzo de producción sin filtro aprendido, con pesos iguales en metal
# recuperado y energía normalizados.
plain = optimize(case, case.nominal, contract, weight=0.5, path=False, screen=None)
print("Estado sin filtro:", plain["status"])
print("Evaluaciones sin filtro:", plain["evaluations"])
print("Motivos de detención:", [r["stop"] for r in plain["starts"]])
plain_end = plain.get("optimum") or plain.get("least_violating") or plain["base"]
display(pd.Series(plain_end["slacks"], name="Holgura de restricción").to_frame())


Auditar el objetivo inicial mediante otra resolución de circuito e informar disponibilidad o código de rechazo. Esa llamada adicional no integra candidatos; la factibilidad sigue dependiendo de holguras informadas.

In [ ]:
# Audit the reported unscreened endpoint only when the engine can define that state.
from pipeline.engine.model import InfeasibleState

plain_point = case.nominal.with_values(**plain_end["decisions"])
plain_check = None
if plain["status"] != "base_refused":
    try:
        plain_check = simulate(case.ore, case.plant, plain_point)
    except InfeasibleState as refusal:
        print("Rechazo de auditoría sin filtro:", refusal.code)
print("Auditoría sin filtro disponible:", plain_check is not None)
if plain_check is not None:
    print("Recuperación exacta sin filtro (%):", round(plain_check.metrics["recovery_pct"], 3))
    print(
        "Energía exacta sin filtro (kWh/t):",
        round(plain_check.metrics["specific_energy_total_kwh_t"], 3),
    )


Dibujar historia incumbente y malla del comienzo seleccionado. El conteo horizontal del primer panel registra evaluaciones exactas dentro de ese comienzo. El segundo registra iteraciones, no llamadas al motor. Contraer malla indica una exploración local más fina; un objetivo factible plano puede coexistir con intentos de mejorar un punto infactible.

In [ ]:
# Graficar el objetivo incumbente y la malla del comienzo seleccionado sin filtro.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 8})
history = pd.DataFrame(
    plain.get("trace", []), columns=["Outcome", "Mesh", "Barrier", "Objective", "Evaluations"]
)
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
axes[0].plot(history["Evaluations"], history["Objective"], marker=".")
axes[0].set(xlabel="Llamadas exactas (n)", ylabel="Objetivo (1)", title="Progreso del objetivo")
axes[1].step(range(len(history)), history["Mesh"], where="post")
axes[1].set(xlabel="Iteración (n)", ylabel="Malla (1)", title="Malla de búsqueda")
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


La historia explica cómo se alcanzó la configuración terminal. Saltos son mejoras aceptadas; mesetas pueden reflejar rechazos o trabajo de restricciones. Aquí se traza solo un comienzo y el conteo impreso cubre toda la corrida. Interpretar la última malla junto al motivo de detención y recuerda las escalas nominales al comparar objetivo, recuperación y energía.

Holgura positiva de potencia significa que la capacidad supera demanda; holgura positiva de ley significa cumplir especificación. Tienen unidades distintas y la barrera usa contrapartes normalizadas, sin sumar valores crudos. El objetivo metal-energía tiene sus propias escalas nominales. Separar normalización de factibilidad y preferencias evita interpretar un margen grande de potencia como motivo para que mejore el objetivo. Una holgura pequeña tampoco mide probabilidad de incumplir: la búsqueda es determinista y no propaga distribuciones minerales en esta lección.

Relacionar molienda inicial con estado factible, menor violación o rechazo, holguras, llamadas y detenciones. La convergencia se muestra como informada, sin inferirla del nombre del extremo. Esta evidencia precede calificar la configuración como factible.

In [ ]:
# Informar cada corrida antes de comparar costo o preferencias.
runs = {"Sin filtro": plain}
endpoint_rows = []
for label, run in runs.items():
    end = run.get("optimum") or run.get("least_violating") or run["base"]
    physical = run["status"] != "base_refused" and bool(end["values"])
    endpoint_rows.append(
        {
            "Corrida": label,
            "Estado": run["status"],
            "Factible": end["feasible"],
            "Convergencia": run.get("converged"),
            "Detenciones": ", ".join((r["stop"] for r in run["starts"])),
            "Decisiones": end["decisions"],
            "Holguras": end["slacks"],
            "Llamadas exactas": run["evaluations"],
            "Metal recuperado (t/h)": end["recovered_tph"] if physical else None,
            "Recuperación (%)": end["recovery_pct"] if physical else None,
            "Energía específica (kWh/t)": end["values"].get("energy_kwh_t") if physical else None,
        }
    )
endpoints = pd.DataFrame(endpoint_rows)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(endpoints)


## Resultados

**Respuesta:** La corrida inicial informa objetivo <!-- result:P80 objetivo informado (µm) -->0<!-- /result --> µm, factibilidad <!-- result:Extremo factible (1=sí) -->0<!-- /result --> y convergencia de malla informada <!-- result:Convergencia informada (1=sí) -->0<!-- /result --> (1=sí), con <!-- result:Conteo sin filtro -->0<!-- /result --> llamadas exactas. La tabla identifica extremo admisible, menor violación o base rechazada; informar un objetivo no certifica factibilidad.

Resumir objetivo informado, factibilidad, convergencia y costo exacto sin inventar valores físicos ausentes.

In [ ]:
# Resumir objetivo informado, factibilidad, convergencia y costo exacto sin inventar valores
# físicos ausentes.
answer_values = {
    "Conteo sin filtro": plain["evaluations"],
    "P80 objetivo informado (µm)": plain_end["decisions"]["target_p80_um"],
    "Extremo factible (1=sí)": int(plain_end["feasible"]),
    "Convergencia informada (1=sí)": int(plain.get("converged") is True),
}
answer_values = {label: round(float(number), 8) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

La molienda informada tiene sentido con estado y holguras. La traza explica detención de producción y la auditoría distingue un estado definido de una base rechazada. El conteo describe solo trabajo de búsqueda.

## Referencias

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662